# Временные диаграммы персистентности — отчёт Результата 2

**Задача 2 первого этапа.** Подпакет `tdadyn.zigzag`: зигзаг-персистентность на
динамических графах и локализация топологических событий во времени.

Критерий из заявки: корректное обнаружение **заранее известных** событий — рождения,
слияния, исчезновения циклов — на синтетических графах с контролируемой динамикой; метрики
recall и precision событий и ошибка локализации.

Отчёт не вычисляет ничего. Он читает артефакт, который производит одна команда:

```bash
python -m tdadyn.results --config config.yaml --results
```

Поэтому число в прозе и число в артефакте не могут разойтись.

In [1]:
import json, pathlib
ART = json.loads(
    (pathlib.Path("..") / "data" / "synthetic" / "results" / "zigzag_events.json")
    .read_text(encoding="utf-8"))
CLEAN = {e["dataset_id"].split("_N")[0]: e for e in ART["datasets"] if e["kind"] == "clean"}
NOISY = [e for e in ART["datasets"] if e["kind"] == "noisy"]
print(f"артефакт версии {ART['format_version']}, собран {ART['generated_at']}")
print(f"пакет {ART['package_version']}, sha256 {ART['content_sha256'][:16]}...")
print(f"наборов {len(ART['datasets'])}: {len(CLEAN)} чистых, {len(NOISY)} зашумлённых")

артефакт версии 1, собран 2026-08-22T22:10:50Z
пакет 0.1.0, sha256 113f44db8e18e41b...
наборов 18: 2 чистых, 16 зашумлённых


---

## 1. Что именно проверяется

Источник истины — семейство 2 из Задачи 5: две последовательности снимков графа, в которых
события **запланированы конструкцией**, а не найдены алгоритмом.

| Набор | События | Чем труден |
|---|---|---|
| `cycle_lifecycle` | 5: рождения 12 и 27 в $H_1$, гибель 41 в $H_1$, рождение 48 и гибель 55 в $H_0$ | комбинаторный, повтор состояния намеренный |
| `geometric_loop` | 2: гибель 20 и рождение 45 в $H_1$ | **метрическая** фильтрация, дыхание радиуса не порождает событий |

Зигзаг строится над множеством индексов $I$, где между снимками стоят объединения. Это не
деталь записи: барштрих-коды, полученные при разных конвенциях объединения, **несравнимы**,
поэтому конвенция публикуется вместе с ними.

In [2]:
c = ART["convention"]
print(f"правило объединения : {c['union_rule']}")
print(f"поле коэффициентов  : Z/{c['coeff_field']}")
print(f"размерность         : до H_{c['max_homology_dim']}")
print(f"бэкенд              : {c['backend']}")
print(f"допуск сопоставления: tau = {c['event_matching']['tolerance']}")
print()
print(c["index_convention"])

правило объединения : complex_union
поле коэффициентов  : Z/2
размерность         : до H_1
бэкенд              : dionysus
допуск сопоставления: tau = 0

barcodes live on I = {0, 1/2, 1, ..., T-1}: integer positions are snapshots, half-integer ones the unions between them. An event is placed on the first snapshot index at or after a birth and strictly after a death (docs/zigzag.md 4.2).


---

## 2. Порядок работ: оракул до бэкенда

Нерушимое место всей задачи. Если сперва завести библиотеку зигзаг-персистентности, она
станет собственным эталоном, и первая же ошибка перевода индексов закрепится как истина.

Поэтому сначала построен **независимый оракул**: он вычисляет барштрих-код другим способом —
через обобщённый ранг $r(b,d) = \operatorname{rank}(\varprojlim \to \varinjlim)$ и
обращение Мёбиуса, — без единой библиотеки персистентности. Только потом появился Dionysus.

Второй бэкенд проверкой не является: он повторяет тот же класс алгоритма и при расхождении
не скажет, кто прав. Проверкой является оракул.

In [3]:
a = ART["agreement"]
print("сверка оракул <-> Dionysus, бар в бар с кратностями:")
for name in a["datasets"]:
    print(f"  {name.split('_N')[0]}")
print(f"  сверено баров: {a['bars_compared']}")
print(f"  расхождений  : {a['discrepancies']}")

сверка оракул <-> Dionysus, бар в бар с кратностями:
  cycle_lifecycle
  geometric_loop
  сверено баров: 7
  расхождений  : 0


### 2.1. Пара 12↔41

Гибель на снимке 41 спарена с рождением на 12, а не с рождением на 27. Это **не конвенция**:
разложение зигзаг-модуля на интервалы однозначно (Габриэль, Крулль–Шмидт), и умирает класс,
рождённый при замыкании пути.

Интуиция здесь подводит: «хорда породила два цикла, ломаем один — умирает молодой» неверно.
Ответ зависит от того, какое ребро удалено, то есть определён геометрией.

In [4]:
for name, e in CLEAN.items():
    print(f"{name}:")
    for dim, bars in sorted(e["barcode"].items()):
        pretty = ", ".join(f"[{b}, {d}]" for b, d in bars)
        print(f"  H_{dim} (позиции на I): {pretty}")

cycle_lifecycle:
  H_0 (позиции на I): [0, 118], [96, 108]
  H_1 (позиции на I): [23, 81], [53, 118]
geometric_loop:
  H_0 (позиции на I): [0, 118]
  H_1 (позиции на I): [0, 39], [89, 118]


Полуцелые концы — не погрешность. Замыкание пути на снимке 12 есть **добавление** ребра,
поэтому объединение $K_{11\frac12}$ совпадает с $K_{12}$ и класс существует уже там.
Удаление ребра на 41 — наоборот, поэтому умирающий класс доживает до $K_{40\frac12}$.
У $H_0$ знаки противоположны, и там концы целые.

Прежняя редакция спецификации записывала эти бары снимковыми индексами и была неверна на
полшага; исправлено замером.

---

## 3. Результат 2: метрики

Перевод бара в событие — два асимметричных правила: рождение попадает на первый снимковый
индекс **не раньше** рождения бара, гибель — на первый снимковый **строго позже** его
гибели. Привычный сдвиг $d \mapsto d+1$ есть вторая половина правила при целом конце.

In [5]:
print(f"{'набор':<18}{'recall':<9}{'precision':<12}{'|dt| max':<10}{'по размерностям'}")
for name, e in CLEAN.items():
    m = e["metrics"]
    per = "  ".join(f"H_{d}: {v['n_matched']}/{v['n_truth']}"
                    for d, v in sorted(m["by_dimension"].items()) if v["n_truth"])
    print(f"{name:<18}{m['recall']:<9.3f}{m['precision']:<12.3f}"
          f"{m['max_localisation_error']:<10}{per}")

набор             recall   precision   |dt| max  по размерностям
cycle_lifecycle   1.000    1.000       0         H_0: 2/2  H_1: 3/3
geometric_loop    1.000    1.000       0         H_1: 2/2


**Единицы — ожидаемый исход, а не достижение.** Наборы построены так, что ответ известен
заранее, а алгоритм точен. Ценность метрики в том, что под шумом она единицей быть
перестаёт (раздел 5).

Ошибка локализации при $\tau = 0$ тождественно нулевая: сматчиться может только точное
совпадение. Она публикуется, потому что её требует спецификация, но информативными
величинами являются recall и precision. Замерено: на восьми зашумлённых наборах поднятие
допуска до единицы возвращает одно событие в одном из них — под шумом отказ есть пропуск и
наводнение ложными, а не смещение.

### 3.1. Чего Результат 2 **не** проверяет

Это надо сказать в самом отчёте, а не оставить читателю.

Событие не помнит, из какого бара оно выведено. Поэтому верная раскладка $H_1$ и
перекрёстная дают **один и тот же лог событий**, и recall с precision пару 12↔41 не
проверяют. Её проверяет сверка барштрих-кодов из раздела 2.

Это третий уровень одной и той же слепоты: кривая Бетти слепа к спариванию, покрытия вершин
и стрелок слепы к спариванию, лог событий слеп к спариванию. Каждый следующий уровень
отчётности теряет его заново.

---

## 4. Строки отчёта: что считается и протоколируется

Четыре величины, которые не отбрасываются молча, потому что каждая — характеристика
конструкции, а не шум реализации.

In [6]:
print(f"{'набор':<18}{'невидимых':<11}{'цензур.':<9}{'из полуцелого':<15}"
      f"{'эфемерных (пролог + вне)'}")
for name, e in CLEAN.items():
    k = e["counters"]
    print(f"{name:<18}{k['invisible_bars']:<11}{k['censored_ends']:<9}"
          f"{k['events_from_half_integer_ends']:<15}"
          f"{k['ephemeral_bars']} = {k['ephemeral_in_prologue']} + "
          f"{k['ephemeral_outside_prologue']}")

набор             невидимых  цензур.  из полуцелого  эфемерных (пролог + вне)
cycle_lifecycle   0          3        3              23 = 23 + 0
geometric_loop    0          4        2              165 = 71 + 94


**Невидимый бар** не содержит в носителе ни одного снимкового индекса: класс не существует
ни в одном состоянии данных. На чистых наборах их ноль — и это замер, а не умолчание.

**Эфемерный бар** рождается и умирает внутри одного перехода. Грубому пути и оракулу они
недоступны по построению; видны только на тонком, где бэкенду даётся по одному времени на
операцию. Число публикуется **разложенным**: прологовая часть измеряет порядок сборки
первого комплекса — произвольный, зафиксированный лексикографически, — а не данные.

---

## 5. Нулевая модель: измерение, а не проверка на ноль

У нулевой модели расписания нет и лог событий пуст, поэтому **любое** событие есть ложное
срабатывание. Правильный ответ — не «ноль срабатываний», а частота с числом испытаний и
доверительным интервалом. Урок S8: три сида, давшие ноль, ограничивают частоту 63
процентами, а не нулём.

In [7]:
n = ART["null_model"]
print(f"ER при плотности cycle_lifecycle, {len(n['seeds'])} сида x "
      f"{n['realizations']} реализаций")
for key, label in (("per_snapshot", "на снимок"), ("per_realisation", "на реализацию")):
    r = n[key]
    lo, hi = r["interval"]
    print(f"  {label:<14}{r['rate']:.4f}  ({r['hits']}/{r['trials']}, "
          f"95% Уилсон [{lo:.4f}, {hi:.4f}])")
print(f"  событий: {n['events']} ({', '.join(f'H_{d}: {c}' for d, c in sorted(n['events_by_dimension'].items()))})")
print(f"  на снимок: {n['events_per_snapshot']:.1f}")

ER при плотности cycle_lifecycle, 3 сида x 20 реализаций
  на снимок     0.9833  (3540/3600, 95% Уилсон [0.9786, 0.9870])
  на реализацию 1.0000  (60/60, 95% Уилсон [0.9398, 1.0000])
  событий: 46323 (H_0: 25710, H_1: 20613)
  на снимок: 12.9


**Каждый снимок нулевой модели есть топологическое событие.** Это не дефект алгоритма:
случайный граф при этой плотности честно имеет $\beta_0 = 4\ldots5$ и
$\beta_1 = 2\ldots5$ на каждом снимке, а рёбра перерисовываются независимо, поэтому классы
рождаются и умирают непрерывно.

Публикуются **оба знаменателя**, потому что ни один в отдельности не честен. На снимок —
испытания коррелированы, соседние комплексы делят рёбра, и интервал уже истинного. На
реализацию — испытания независимы, но их шестьдесят, и интервал широк. Третьей строкой идёт
число событий на снимок: частота 0.98 насыщена и о масштабе не говорит.

Порог `null_models.max_false_positive_rate` сюда **не применяется**: он откалиброван для
критерия значимости на облаках семейства 1 — доля реализаций, где критерий выдумал признак, —
а не для доли снимков со срабатыванием.

---

## 6. Деградация под шумом

In [8]:
rows = sorted(NOISY, key=lambda e: (e["noise"]["kind"], e["base"], e["noise"]["level"]))
print(f"{'механизм':<10}{'база':<17}{'ур.':<7}{'баров':<8}{'recall':<9}"
      f"{'precision':<12}{'невид.':<8}{'эфем. вне пролога'}")
for e in rows:
    m, k = e["metrics"], e["counters"]
    bars = sum(len(v) for v in e["barcode"].values())
    print(f"{e['noise']['kind'][:9]:<10}{e['base'].split('_N')[0]:<17}"
          f"{e['noise']['level']:<7}{bars:<8}{m['recall']:<9.3f}{m['precision']:<12.4f}"
          f"{k['invisible_bars']:<8}{k['ephemeral_outside_prologue']}")

механизм  база             ур.    баров   recall   precision   невид.  эфем. вне пролога
missing_e cycle_lifecycle  0.0    4       1.000    1.0000      0       0
missing_e cycle_lifecycle  0.02   4       1.000    1.0000      0       0
missing_e cycle_lifecycle  0.05   61      0.800    0.0506      19      0
missing_e cycle_lifecycle  0.1    113     0.600    0.0201      36      0
missing_e geometric_loop   0.0    3       1.000    1.0000      0       94
missing_e geometric_loop   0.02   28      1.000    0.0385      0       200
missing_e geometric_loop   0.05   50      1.000    0.0208      0       336
missing_e geometric_loop   0.1    97      1.000    0.0109      3       531
spurious_ cycle_lifecycle  0.0    4       1.000    1.0000      0       0
spurious_ cycle_lifecycle  0.02   269     0.800    0.0078      6       54
spurious_ cycle_lifecycle  0.05   567     0.600    0.0027      8       334
spurious_ cycle_lifecycle  0.1    915     0.600    0.0017      4       857
spurious_ geometric_loo

Три наблюдения.

**recall падает медленно, precision обваливается.** У `cycle_lifecycle` точность доходит до
0.17%: алгоритм не теряет события так быстро, как наводняет лог ложными.

**`geometric_loop` держит recall на единице при всех уровнях.** Оба его события находятся
всегда; портится только точность. Это свойство наборов, а не шум измерения.

**`spurious_edges` вреднее `missing_edges` на порядок.** Добавленное ребро создаёт циклы;
удалённое чаще всего лишь разрывает существующий.

Уровень 0.0 воспроизводит чистый набор точно — бесплатная проверка, ловящая разом неверно
подключённую базу, сдвиг потока случайности между уровнями и безусловный `clip`.

---

## 7. Границы применимости

- **Барштрих-код зависит от конвенции объединения.** Коды, полученные при разных
  конвенциях, несравнимы; конвенция указана в разделе 1 и лежит в артефакте.
- **Обрезка комплекса.** Бары размерности $k_{\max}$ достоверны; размерность выше не
  вычисляется, и её отсутствие в выводе не есть утверждение о её отсутствии в данных.
  Бэкенд возвращает такие бары (69 штук на метрическом наборе), и адаптер их отсекает:
  без $\partial_{k+2}$ каждая замкнутая цепь верхней размерности считается полостью.
- **Значимость бара в зигзаге не определена**, и замена критерию семейства 1 в этой ветке
  не вводится. Раздел 5 показывает, что вопрос не академический: без критерия точность на
  шуме падает до 0.17%, а на нулевой модели срабатывает 98% снимков.
- **Параметры `topology_check` не наследуются.** Они откалиброваны на облаках семейства 1, и
  критерий, кроме того, не инвариантен к перепараметризации фильтрации.

## 8. Что дальше

| | |
|---|---|
| **Vineyards** | отслеживают перемещение точек диаграммы при **непрерывном** изменении фильтрации; зигзаг здесь дискретен по построению, между целым и полуцелым индексом ничего нет. Нужны там, где фильтрация непрерывна — Задача 3 и бифильтрация Результата 3 |
| **Значимость бара** | ставится заново, и не подбором порога под precision |
| **Коннектомы** | числа невидимых и эфемерных баров из раздела 4 — то, что надо знать до переноса |